In [2]:
# Importing of required items
import pandas as pd
import itertools
import csv

In [3]:
# Read the file that we will use and assign them variables name
m_seeds = pd.read_csv('MNCAATourneySeeds.csv')
w_seeds = pd.read_csv('WNCAATourneySeeds.csv')
m_games = pd.read_csv('MNCAATourneyCompactResults.csv')
w_games = pd.read_csv('WNCAATourneyCompactResults.csv')
print(m_seeds.head())
print(w_seeds.head())

   Season Seed  TeamID
0    1985  W01    1207
1    1985  W02    1210
2    1985  W03    1228
3    1985  W04    1260
4    1985  W05    1374
   Season Seed  TeamID
0    1998  W01    3330
1    1998  W02    3163
2    1998  W03    3112
3    1998  W04    3301
4    1998  W05    3272


In [4]:
# print(m_games['Season'].max())
# print(m_games['Season'].min())

In [5]:
# Calculating the weights of the men  and women
m_games['Weight'] = m_games['Season'] - m_games['Season'].min() + 1
w_games['Weight'] = w_games['Season'] - w_games['Season'].min() + 1

In [ ]:
# Calculating all the wins and lose of the teams then the win rates of the teams
m_win_counts = m_games.groupby('WTeamID')['Weight'].sum() 
m_lose_counts = m_games.groupby('LTeamID')['Weight'].sum()
m_total_counts = (m_win_counts + m_lose_counts).fillna(0)
m_win_rates = (m_win_counts/ m_total_counts).fillna(0.5)
m_win_rates = m_win_rates.replace(0, 0.5)
print(m_win_counts.head())
print(m_win_rates.head())



# Calculating all the win and lose of women, then the win rate of the teams 
w_win_counts = w_games.groupby('WTeamID')['Weight'].sum()
w_lose_counts = w_games.groupby('LTeamID')['Weight'].sum()
# print(m_lose_counts)
# print(m_win_counts)
w_total_counts = (w_win_counts + w_lose_counts).fillna(0)
w_win_rates = (w_win_counts/w_total_counts).fillna(0.5)
w_win_rates = w_win_rates.replace(0.0, 0.5)
# print(total_counts.head())
# print(w_win_rates)



In [ ]:
m_seeds['Seed'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['Seed'] = w_seeds['Seed'].str[1:3].astype(int)
# print(m_seeds.head())
print(w_seeds.head())

In [ ]:
# We get the latest seed of each team
m_seeds_latest = m_seeds[m_seeds['Season'] == m_seeds['Season'].max()]
w_seeds_latest = w_seeds[w_seeds['Season'] == w_seeds['Season'].max()] 

print(m_seeds_latest.head())
# print(m_seeds.head())
print(w_seeds_latest.head())
# print(w_seeds.head())
# print(m_seeds['Seed'].unique())
# print(w_seeds['Seed'].unique())


In [9]:
# Generates the matchups of the men's and women's teams
all_m_matchups = (list(itertools.combinations(m_win_rates.index, 2)))
all_w_matchups = (list(itertools.combinations(w_win_rates.index, 2)))
# print(all_m_matchups)
# print(len(all_w_matchups))

In [10]:
# Convert the TeamID from interger so as to combine with m_seed_latest
m_win_rates_df = m_win_rates.reset_index()
m_win_rates_df.columns = ['TeamID', 'WinRate']

w_win_rates_df = w_win_rates.reset_index()
w_win_rates_df.columns = ['TeamID', 'WinRate']

# print(w_win_rates_df.head())
# print(m_win_rates_df.head())

In [11]:
# Calculating the strength
m_seeds_latest['Strength'] = (17 - m_seeds_latest['Seed'])
# Combining the m_seed_latest and m_win_rate_df
m_team_stats = m_win_rates_df.merge(m_seeds_latest, on='TeamID', how='left')
# Fill unknown teams with neautral strength
m_team_stats['Strength']  = m_team_stats['Strength'].fillna(9)
m_team_stats['Score'] = m_team_stats['WinRate'] * m_team_stats['Strength']

w_seeds_latest['Strength'] = (17 - w_seeds_latest['Seed'])
w_team_stats = w_win_rates_df.merge(w_seeds_latest, on='TeamID', how='left')
w_team_stats['Strength'] = w_team_stats['Strength'].fillna(9)
w_team_stats['Score'] = w_team_stats['WinRate'] * w_team_stats['Strength']
# print(m_team_stats.head(10))
# print(w_team_stats.head(10))
# print(len(m_team_stats))
# print(m_team_stats[m_team_stats['TeamID'] == 1181])

In [12]:
# Calculating the probability between the two teams
m_row = []
for (team1, team2) in all_m_matchups:
    m_score1 = m_team_stats[m_team_stats['TeamID'] == team1]['Score'].values[0]
    m_score2 = m_team_stats[m_team_stats['TeamID'] ==  team2]['Score'].values[0]
    propablity = m_score1 / (m_score1 +  m_score2)
    m_match_ids = f"2026_{team1}_{team2}"
    m_row.append([m_match_ids, propablity])

w_row= []
for (team1, team2) in all_w_matchups:
    w_score1 = w_team_stats[w_team_stats['TeamID'] == team1]['Score'].values[0]
    w_score2 = w_team_stats[w_team_stats['TeamID'] == team2]['Score'].values[0]
    propablity = w_score1 / (w_score1 + w_score2)
    w_match_ids = f"2026_{team1}_{team2}"
    w_row.append([w_match_ids, propablity])
    # print(w_row)

In [13]:
# print(m_games[m_games['WTeamID'] == 1181]['WTeamID'].count())
# print(m_win_counts[1181])

In [15]:
combined_pred = w_row + m_row

df = pd.DataFrame(combined_pred, columns=['ID', 'Pred'])

df.to_csv('matchup_probabilities.csv', index=False)
print(df.head())
print(len(df))
print(df.isnull().sum())
print(df['Pred'].max())
print(df['Pred'].min())
print(df[df['ID'].str.contains('_3')].head())
print(df[df['ID'].str.contains('_1')].head())


               ID      Pred
0  2026_3101_3103  0.500000
1  2026_3101_3104  0.463217
2  2026_3101_3106  0.500000
3  2026_3101_3107  0.796791
4  2026_3101_3108  0.500000
89298
ID      0
Pred    0
dtype: int64
0.9934869040560806
0.006443632865759566
               ID      Pred
0  2026_3101_3103  0.500000
1  2026_3101_3104  0.463217
2  2026_3101_3106  0.500000
3  2026_3101_3107  0.796791
4  2026_3101_3108  0.500000
                   ID      Pred
40470  2026_1101_1102  0.404372
40471  2026_1101_1103  0.604356
40472  2026_1101_1104  0.250917
40473  2026_1101_1105  0.404372
40474  2026_1101_1106  0.927229


In [ ]:
# print(m_win_rates[1207])

In [ ]:
# print(m_seeds_latest.head())
# print(m_win_rates_df.head())